In [1]:
#!/usr/bin/env python3
"""
train.py
========
Train the dynamic-sign LSTM on dataset.npz (from preprocess.py), evaluate it,
and export sign_model.tflite for the Flutter app.

Outputs:
    sign_model.keras        - the trained Keras model (reference / re-training)
    sign_model.tflite       - the on-device model the phone runs
    confusion_matrix.png     - per-sign accuracy picture (great for your README)

------------------------------------------------------------------------------
SETUP  (TensorFlow is a big download -- this is the heavy install)
------------------------------------------------------------------------------
    pip install tensorflow numpy
    pip install matplotlib        # optional, only for the confusion-matrix image

Then:
    python train.py
"""

import json
from pathlib import Path

import numpy as np
import tensorflow as tf

# ---------------------------- CONFIG ----------------------------------------
DATASET = Path("dataset.npz")
LABELS = Path("labels.json")
KERAS_OUT = Path("sign_model.keras")
TFLITE_OUT = Path("sign_model.tflite")
CM_PNG = Path("confusion_matrix.png")

EPOCHS = 200
BATCH = 16
PATIENCE = 25          # early-stopping patience on val_loss
VAL_SPLIT = 0.2
SEED = 42
# ----------------------------------------------------------------------------


def build_model(seq_len, n_features, n_classes):
    tf.random.set_seed(SEED)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(seq_len, n_features)),
        # unroll=True is important: with a fixed sequence length it lets the
        # LSTM convert to TFLite *builtin* ops, so the model runs on stock
        # tflite_flutter with NO Flex delegate / extra setup on the phone.
        tf.keras.layers.LSTM(64, return_sequences=True, unroll=True),
        tf.keras.layers.LSTM(128, unroll=True),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(n_classes, activation="softmax"),
    ], name="sign_lstm")
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model


def confusion_matrix(y_true, y_pred, n):
    cm = np.zeros((n, n), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def print_report(cm, labels):
    print("\nPer-sign accuracy:")
    for i, name in enumerate(labels):
        total = cm[i].sum()
        acc = cm[i, i] / total if total else 0.0
        print(f"  {name:<12} {acc*100:5.1f}%   ({cm[i, i]}/{total})")
    print("\nConfusion matrix (rows = true, cols = predicted):")
    header = "            " + " ".join(f"{n[:6]:>7}" for n in labels)
    print(header)
    for i, name in enumerate(labels):
        row = " ".join(f"{v:>7}" for v in cm[i])
        print(f"  {name[:10]:<10} {row}")


def save_confusion_png(cm, labels):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("(matplotlib not installed -- skipping confusion_matrix.png)")
        return
    fig, ax = plt.subplots(figsize=(1.2 * len(labels) + 2,
                                    1.2 * len(labels) + 2))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    thresh = cm.max() / 2 if cm.max() else 0
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.colorbar(im); fig.tight_layout()
    fig.savefig(CM_PNG, dpi=130)
    print(f"Saved {CM_PNG}")


def export_tflite(model):
    """Convert to TFLite. Prefer builtin ops (stock tflite_flutter runtime).
    Fall back to Select-TF ops if the LSTM won't fuse (needs the Flex delegate
    on the Flutter side)."""
    def convert(select_ops):
        c = tf.lite.TFLiteConverter.from_keras_model(model)
        if select_ops:
            c.target_spec.supported_ops = [
                tf.lite.OpsSet.TFLITE_BUILTINS,
                tf.lite.OpsSet.SELECT_TF_OPS,
            ]
            c._experimental_lower_tensor_list_ops = False
        else:
            c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
        return c.convert()

    try:
        data = convert(select_ops=False)
        print("TFLite: converted with builtin ops only (ideal for mobile).")
    except Exception as e:
        print(f"TFLite: builtin-only conversion failed ({type(e).__name__}); "
              "retrying with Select-TF ops.")
        data = convert(select_ops=True)
        print("NOTE: model uses Select-TF ops -> enable the Flex delegate in the "
              "Flutter app (add the select-tf-ops flag to tflite_flutter).")
    TFLITE_OUT.write_bytes(data)
    print(f"Saved {TFLITE_OUT} ({len(data)/1024:.0f} KB)")


def verify_tflite(X_sample, expected_classes):
    """Load the exported model and run one inference to prove it works."""
    interp = tf.lite.Interpreter(model_path=str(TFLITE_OUT))
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    x = X_sample[None, ...].astype(np.float32)        # (1, 30, 126)
    interp.resize_tensor_input(inp["index"], x.shape)
    interp.allocate_tensors()
    interp.set_tensor(inp["index"], x)
    interp.invoke()
    y = interp.get_tensor(out["index"])
    assert y.shape[-1] == expected_classes, "TFLite output size mismatch!"
    print(f"TFLite sanity check OK -> output shape {y.shape}, "
          f"predicted class {int(np.argmax(y))}")


def main():
    if not DATASET.exists():
        raise SystemExit("dataset.npz not found. Run preprocess.py first.")
    d = np.load(DATASET)
    X_train, X_test = d["X_train"], d["X_test"]
    y_train, y_test = d["y_train"], d["y_test"]
    labels = json.loads(LABELS.read_text())
    n_classes = len(labels)
    seq_len, n_features = X_train.shape[1], X_train.shape[2]
    print(f"Train {X_train.shape}, Test {X_test.shape}, {n_classes} signs: {labels}")

    model = build_model(seq_len, n_features, n_classes)
    model.summary()

    cb = [tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=PATIENCE, restore_best_weights=True)]
    model.fit(X_train, y_train,
              validation_split=VAL_SPLIT,
              epochs=EPOCHS, batch_size=BATCH,
              callbacks=cb, verbose=2)

    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\n=== Test accuracy: {acc*100:.1f}%  (loss {loss:.3f}) ===")

    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    cm = confusion_matrix(y_test, y_pred, n_classes)
    print_report(cm, labels)
    save_confusion_png(cm, labels)

    model.save(KERAS_OUT)
    print(f"\nSaved {KERAS_OUT}")
    export_tflite(model)
    verify_tflite(X_test[0], n_classes)
    print("\nDone. Copy sign_model.tflite + labels.json into the Flutter app's assets/.")


if __name__ == "__main__":
    main()

Train (186, 30, 126), Test (47, 30, 126), 8 signs: ['hello', 'help', 'iloveyou', 'no', 'please', 'sorry', 'thanks', 'yes']


Model: "sign_lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        48,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 156,488 (611.28 KB)

 Trainable params: 156,488 (611.28 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
10/10 - 8s - 819ms/step - accuracy: 0.3446 - loss: 1.9084 - val_accuracy: 0.5789 - val_loss: 1.5373
Epoch 2/200
10/10 - 0s - 22ms/step - accuracy: 0.6824 - loss: 1.2186 - val_accuracy: 0.7368 - val_loss: 0.8690
Epoch 3/200
10/10 - 0s - 21ms/step - accuracy: 0.7365 - loss: 0.7011 - val_accuracy: 0.8421 - val_loss: 0.4970
Epoch 4/200
10/10 - 0s - 20ms/step - accuracy: 0.8176 - loss: 0.4963 - val_accuracy: 0.9211 - val_loss: 0.2903
Epoch 5/200
10/10 - 0s - 21ms/step - accuracy: 0.9189 - loss: 0.2650 - val_accuracy: 0.9474 - val_loss: 0.1824
Epoch 6/200
10/10 - 0s - 21ms/step - accuracy: 0.9459 - loss: 0.1917 - val_accuracy: 0.9474 - val_loss: 0.1360
Epoch 7/200
10/10 - 0s - 21ms/step - accuracy: 0.9595 - loss: 0.1149 - val_accuracy: 0.9474 - val_loss: 0.1344
Epoch 8/200
10/10 - 0s - 21ms/step - accuracy: 0.9730 - loss: 0.1000 - val_accuracy: 0.9211 - val_loss: 0.1699
Epoch 9/200
10/10 - 0s - 21ms/step - accuracy: 0.9459 - loss: 0.1584 - val_accuracy: 1.0000 - val_loss: 0.0585


INFO:tensorflow:Assets written to: C:\Users\aliak\AppData\Local\Temp\tmpnz6tv2iy\assets


Saved artifact at 'C:\Users\aliak\AppData\Local\Temp\tmpnz6tv2iy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 30, 126), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 8), dtype=tf.float32, name=None)
Captures:
  2138670267232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138670263888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138670253504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138670267760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138670509472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138670512112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138670498912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138670509648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138670512640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138670513520: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite: con

c:\Users\aliak\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
